# cv_routes · Segmentation de routes IGN (FLAIR-INC)

> **Environnement cible : ArcGIS Online Notebooks**  
> Toutes les dépendances sont installées via la cellule `[0] Setup` ci-dessous.  
> Aucune configuration locale requise.

| # | Contenu |
|---|---------|
| 0 | Installation + téléchargement modèle |
| 1 | Image libre (upload ou URL) |
| 2 | Tuile GeoTIFF locale |
| 3 | Téléchargement WMS IGN par commune |
| 4 | Traitement commune entière + exports GeoTIFF / GeoPackage |
| 5 | Lecture et analyse des exports |

## [0] Setup — installation & modèle
> **Exécuter une seule fois** par session de notebook.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# [0.1]  Dépendances conda / pip
#        (ArcGIS Online Notebooks supporte les deux)
# ─────────────────────────────────────────────────────────────────────────────
import subprocess, sys

def conda_install(*pkgs):
    subprocess.run(
        ["conda", "install", "-c", "conda-forge", "-y", "--quiet", *pkgs],
        check=True
    )

def pip_install(*pkgs):
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", *pkgs],
        check=True
    )

# Packages disponibles sur conda-forge
conda_install("rasterio", "geopandas", "shapely", "fiona", "huggingface_hub")

# onnxruntime via pip (plus récent sur PyPI)
pip_install("onnxruntime>=1.16")

# ─────────────────────────────────────────────────────────────────────────────
# [0.2]  Installation de la librairie cv_routes depuis GitHub
# ─────────────────────────────────────────────────────────────────────────────
pip_install("git+https://github.com/DIGIT-Seine-Ouest/cv-routes-bitumees.git")

print("✓ Installation terminée")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# [0.3]  Téléchargement du modèle ONNX (HuggingFace Hub → cache local)
# ─────────────────────────────────────────────────────────────────────────────
from cv_routes import FlairModel, download_model

MODEL_PATH = download_model(
    dest="flair_12cl_resnet34_unet.onnx",   # chemin dans le workspace AGOL
    source="hf",
    hf_repo="mandresyandri/road-landcover-detection",
    hf_filename="flair_12cl_resnet34_unet.onnx",
)

model = FlairModel(MODEL_PATH)
print(f"✓ Modèle chargé — {model.NUM_CLASSES} classes")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# [0.4]  Imports globaux (à exécuter après le setup)
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from cv_routes import (
    run_on_image,
    run_on_tile,
    run_and_export_tile,
    run_on_city,
    fetch_tiles,
    flair_stats,
    road_mask_from_flair,
    mask_to_geotiff,
    masks_to_gpkg,
    CITIES,
    FLAIR_CLASS_NAMES,
)

print("Communes GPSO disponibles :", list(CITIES.keys()))

---
## [1] Image libre

Charge n'importe quelle ortho-photo — depuis le workspace AGOL, une URL, ou un upload direct.

In [ ]:
import urllib.request, io

# ── Option A : fichier local dans le workspace AGOL ──────────────────────────
# image = Image.open("/arcgis/home/mon_ortho.png")

# ── Option B : téléchargement depuis une URL publique ────────────────────────
IMAGE_URL = "https://huggingface.co/spaces/mandresyandri/road-landcover-detection/resolve/main/examples/sample.jpg"
with urllib.request.urlopen(IMAGE_URL) as r:
    image = Image.open(io.BytesIO(r.read())).copy()

# ─────────────────────────────────────────────────────────────────────────────
result = run_on_image(model, image)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(result["ortho"]);         axes[0].set_title("Ortho (512×512)")
axes[1].imshow(result["flair_map"]);     axes[1].set_title("FLAIR 12 classes")
axes[2].imshow(result["road_overlay"]);  axes[2].set_title(f"Routes FLAIR — {result['road_pct']:.1f}%")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

print(result["stats"]["summary"])

In [ ]:
# Accès aux données brutes
print("Clés du résultat :", list(result.keys()))
print(f"\nclass_map  : shape={result['class_map'].shape}  dtype={result['class_map'].dtype}")
print(f"road_mask  : shape={result['road_mask'].shape}   valeurs={np.unique(result['road_mask'])}")
print(f"\nStats par classe (> 0.5%) :")
for cls, pct in result["stats"]["classes"].items():
    if pct > 0.5:
        print(f"  {cls:24s}: {pct:5.1f}%")

---
## [2] Tuile GeoTIFF géoréférencée + export

`run_and_export_tile` produit un **GeoTIFF** masque (LZW, EPSG:2154) et un **GeoPackage** vectoriel directement exploitables dans ArcGIS Pro / QGIS.

In [ ]:
import rasterio

# ── Chemin vers une tuile GeoTIFF (workspace AGOL : /arcgis/home/…) ──────────
TILE_PATH   = "/arcgis/home/tiles/MEUDON/block_0000/tile_00000.tif"
EXPORT_DIR  = "/arcgis/home/exports/demo_tuile"

result_tile = run_and_export_tile(model, TILE_PATH, EXPORT_DIR)

# Visualisation
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(result_tile["ortho"]);         axes[0].set_title("Ortho")
axes[1].imshow(result_tile["flair_map"]);     axes[1].set_title("FLAIR 12 classes")
axes[2].imshow(result_tile["road_overlay"]);  axes[2].set_title(f"Routes — {result_tile['road_pct']:.1f}%")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

# Métadonnées géospatiales du GeoTIFF produit
with rasterio.open(result_tile["flair_tif"]) as src:
    print("GeoTIFF produit")
    print(f"  CRS    : {src.crs}")
    print(f"  Bounds : {src.bounds}")
    print(f"  Shape  : {src.shape}")
print(f"  GPKG   : {result_tile['gpkg']}")

---
## [3] Téléchargement WMS IGN par commune

`fetch_tiles` appelle le flux **WMS ORTHOIMAGERY.ORTHOPHOTOS** (IGN / data.geopf.fr), découpe en tuiles 512×512 GeoTIFF géoréférencées et les met en cache local.  
Un second appel retourne le cache immédiatement sans requête WMS.

In [ ]:
COMMUNE   = "MEUDON"                               # voir liste dans [0.4]
TILES_DIR = f"/arcgis/home/tiles/{COMMUNE}"        # workspace AGOL

def on_progress(ratio: float, msg: str) -> None:
    bar = "█" * int(ratio * 20) + "░" * (20 - int(ratio * 20))
    print(f"\r  [{bar}] {ratio*100:5.1f}%  {msg}", end="", flush=True)

tiles = fetch_tiles(COMMUNE, TILES_DIR, on_progress=on_progress)
print(f"\n✓ {len(tiles)} tuile(s) prêtes")

In [ ]:
# Aperçu des 6 premières tuiles téléchargées
from cv_routes.data import load_tile_as_rgb
from pathlib import Path

n_preview = min(6, len(tiles))
fig, axes = plt.subplots(1, n_preview, figsize=(4 * n_preview, 4))
if n_preview == 1:
    axes = [axes]
for ax, tile_path in zip(axes, tiles[:n_preview]):
    ax.imshow(load_tile_as_rgb(tile_path))
    ax.set_title(Path(tile_path).name, fontsize=8)
    ax.axis("off")
plt.suptitle(f"Tuiles WMS IGN — {COMMUNE}", fontsize=12)
plt.tight_layout(); plt.show()

---
## [4] Commune entière — inférence batch + exports

`run_on_city` traite toutes les tuiles en un seul appel et produit :
- Un **GeoPackage** vectoriel global (couche `route_flair`, EPSG:2154)
- Un **ZIP** de tous les masques GeoTIFF individuels

In [ ]:
EXPORT_DIR = f"/arcgis/home/exports/{COMMUNE}"

city_result = run_on_city(
    model,
    COMMUNE,
    tiles,
    export_dir=EXPORT_DIR,
    on_progress=on_progress,
)

print(f"\n{'─'*50}")
print(f"Commune        : {city_result['commune']}")
print(f"Tuiles         : {city_result['n_tiles']}")
print(f"Route (moy.)   : {city_result['road_pct']:.1f}%")
print(f"GeoPackage     : {city_result['gpkg']}")
print(f"ZIP GeoTIFFs   : {city_result['tifs_zip']}")

In [ ]:
# Grille Ortho / FLAIR / Routes — 6 premières tuiles
n_show = min(6, len(city_result["tile_results"]))
fig, axes = plt.subplots(n_show, 3, figsize=(13, 4 * n_show))
if n_show == 1:
    axes = [axes]

for i, tr in enumerate(city_result["tile_results"][:n_show]):
    axes[i][0].imshow(tr["ortho"])
    axes[i][1].imshow(tr["flair_map"])
    axes[i][2].imshow(tr["road_overlay"])
    axes[i][0].set_ylabel(f"tuile {i}", fontsize=9)
    axes[i][2].set_title(f"{tr['road_pct']:.1f}% route", fontsize=9)

axes[0][0].set_title("Ortho")
axes[0][1].set_title("FLAIR 12 classes")
axes[0][2].set_title("Routes (orange)")
for row in axes:
    for ax in row: ax.axis("off")
plt.suptitle(f"{COMMUNE} — inférence FLAIR", fontsize=13, y=1.01)
plt.tight_layout(); plt.show()

---
## [5] Lecture des exports — rasterio / geopandas

Les fichiers sont compatibles ArcGIS Pro, QGIS et tout outil Python géospatial.

In [ ]:
import geopandas as gpd

# ── GeoPackage vectoriel ──────────────────────────────────────────────────────
gdf = gpd.read_file(city_result["gpkg"], layer="route_flair")

surface_ha = gdf.geometry.area.sum() / 1e4
print(f"Polygones route  : {len(gdf)}")
print(f"CRS              : {gdf.crs}")
print(f"Surface totale   : {surface_ha:.2f} ha")

fig, ax = plt.subplots(figsize=(8, 8))
gdf.plot(ax=ax, color="orange", alpha=0.75, edgecolor="none")
ax.set_title(f"Routes FLAIR — {COMMUNE}  ({surface_ha:.1f} ha, EPSG:2154)")
ax.set_xlabel("X Lambert 93 (m)")
ax.set_ylabel("Y Lambert 93 (m)")
plt.tight_layout(); plt.show()

In [ ]:
# ── GeoTIFF masque (premier de la liste) ─────────────────────────────────────
from pathlib import Path
import rasterio

first_tif = next(Path(EXPORT_DIR, "tifs").glob("*.tif"))

with rasterio.open(first_tif) as src:
    mask_data = src.read(1)
    print(f"CRS    : {src.crs}")
    print(f"Bounds : {src.bounds}")
    print(f"Shape  : {src.shape}")
    print(f"% route: {mask_data.mean()*100:.1f}%")

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(mask_data, cmap="Oranges", vmin=0, vmax=1)
ax.set_title(first_tif.name)
ax.axis("off")
plt.tight_layout(); plt.show()